# CASMI26 - CFT (contrastive spectrum -> fingerprint transformer) training

Trains the CFT model on a T4 and writes `/kaggle/working/out_cft/export/` (`cft_default.pt`, `train_result_default.json`,
`eval_cft.json`, `check_summary_cft.txt`, logs).

Settings: **Accelerator = GPU T4**, Internet off, private datasets **casmi-train-pkg** and **casmi-train-cft** attached.
Use **Save Version -> Save & Run All (Commit)** so the run survives closing the browser.
If a session ends before it is finished: add this notebook's own output as an input and run it again - it resumes.

In [ ]:
import glob, os, subprocess, sys
hits = sorted(glob.glob('/kaggle/input/**/train_cft.py', recursive=True))
assert hits, 'attach the private dataset casmi-train-cft (Add Input)'
CFT = os.path.dirname(hits[0]); print('CFT package:', CFT)
base = sorted(glob.glob('/kaggle/input/**/spec_mz.npy', recursive=True))
assert base, 'attach the private dataset casmi-train-pkg (Add Input)'
BASE = os.path.dirname(base[0]); print('base data:', BASE)
PRESET = 'default'          # 'small' = the 4-layer model
prev = sorted(set(os.path.dirname(p) for p in glob.glob(f'/kaggle/input/**/ckpt_{PRESET}.pt', recursive=True)))
print('earlier checkpoints:', prev or 'none')
!nvidia-smi -L
import torch; print(torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'turn on the GPU: Settings -> Accelerator -> GPU T4'

In [ ]:
# 600 minutes keeps training + evaluation inside the 12-hour session limit; a second run resumes automatically.
cmd = [sys.executable, f'{CFT}/train_cft.py', '--data', BASE, CFT, '--out', '/kaggle/working/out_cft',
       '--preset', PRESET, '--max_minutes', '600']
if prev: cmd += ['--resume_from', prev[0]]
rc = subprocess.call(cmd); print('exit code', rc, '(0 = finished, 2 = not finished yet: run again, 3 = out of GPU memory)')

In [ ]:
base_dirs = sorted(set(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/fp_single*.pt', recursive=True)))
cmd = [sys.executable, f'{CFT}/eval_cft.py', '--data', BASE, CFT, '--out', '/kaggle/working/out_cft']
if base_dirs: cmd += ['--baseline', base_dirs[0]]      # warm-up networks, if their output is attached
rc2 = subprocess.call(cmd)
print(open('/kaggle/working/out_cft/export/check_summary_cft.txt').read())
!ls -la /kaggle/working/out_cft /kaggle/working/out_cft/export